<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 07 · Object-Oriented Programming

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the objects are defined in
  order.
- Add your own cells for experiments, tests, or refactorings.
- Compare the code with the book text when you want the surrounding
  explanation.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

This chapter shows why classes are useful once finance code needs reusable
state and behavior.


# Why Object Orientation Matters in Finance


Grouping related data and methods makes finance code easier to extend, test,
and reuse.


# Defining Simple Classes


Start with a tiny class that stores a symbol, quantity, and price.


In [ ]:
class Position:  # Define a `Position` class using the `class` keyword.
    def __init__(self, symbol: str, qty: float, price: float) -> None:
        self.symbol = symbol  # Store the asset symbol as an instance attribute.
        self.qty = qty  # Store the quantity (for example, number of shares).
        self.price = price  # Store the current price per unit.

In [ ]:
# Create an instance by calling the class like a function and passing
# constructor arguments.
pos = Position("AAPL", 10, 180.0)

In [ ]:
pos.symbol, pos.qty, pos.price

Add methods to move behavior next to the data they operate on.


In [ ]:
class Position:  # Redefine `Position` with the same attributes as before.
    def __init__(self, symbol: str, qty: float, price: float) -> None:
        self.symbol = symbol
        self.qty = qty
        self.price = price
    # Add a method that computes the current market value of the position.
    def market_value(self) -> float:
        return self.qty * self.price

In [ ]:
pos = Position("AAPL", 10, 180.0)

In [ ]:
# Call the method on an instance; Python passes `self` automatically.
pos.market_value()


# Special Methods: `__init__` and `__repr__`


Special methods let your objects print cleanly and behave naturally in the
interpreter.


### Improving Debugging with `__repr__`


A useful `__repr__` helps you inspect objects quickly while debugging.


In [ ]:
class Position:
    def __init__(self, symbol: str, qty: float, price: float) -> None:
        self.symbol = symbol
        self.qty = qty
        self.price = price

    def market_value(self) -> float:
        return self.qty * self.price

    def __repr__(self) -> str:
        return (
            "Position("
            f"symbol={self.symbol!r}, "
            f"qty={self.qty}, "
            f"price={self.price}"
            ")"
        )


In [ ]:
# In the REPL, the object now displays with its key fields, which makes
# debugging easier.
Position("AAPL", 10, 180.0)

### Using Dataclasses for Boilerplate


Dataclasses remove a lot of manual boilerplate when the class mainly stores
data.


In [ ]:
from dataclasses import dataclass

In [ ]:
# The `@dataclass` decorator asks Python to generate `+.__init__()+` and
# `+.__repr__()+` based on the annotated fields.
@dataclass
class Position:
    symbol: str
    qty: float
    price: float
    # You can still define custom methods alongside the generated boilerplate.
    def market_value(self) -> float:
        return self.qty * self.price

In [ ]:
pos = Position("AAPL", 10, 180.0)

In [ ]:
# The REPL representation is helpful by default, without writing
# `+.__repr__()+` manually.
pos

# Modeling Portfolios with Classes


Portfolios work well as small objects with methods for the values you need
often.


### A Simple Portfolio Class


A portfolio can aggregate the market value of its positions.


In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Position:
    symbol: str
    qty: float
    price: float
    def market_value(self) -> float:
        return self.qty * self.price

In [ ]:
class Portfolio:  # Define a `Portfolio` class that groups positions together.
    def __init__(self, positions: list[Position]) -> None:
        # Store a list of `Position` instances as an attribute.
        self.positions = positions
    # Provide a method that computes the total market value by
    # delegating to each
    # position's `.market_value()` method.
    def total_value(self) -> float:
        return sum(p.market_value() for p in self.positions)

In [ ]:
portfolio = Portfolio(
    [Position("AAPL", 10, 180.0), Position("MSFT", 5, 350.0)]
)

In [ ]:
portfolio.total_value()

The assertion below checks the implementation against a hand-computed total.


In [ ]:
# Quick self-check that raises an error immediately if either
# `.market_value()` or `.total_value()` changes in an unintended way.
assert portfolio.total_value() == 10 * 180.0 + 5 * 350.0

### Adding Convenience Methods


Convenience methods can express intent directly and keep calling code
readable.


In [ ]:
class Portfolio:
    def __init__(self, positions: list[Position]) -> None:
        self.positions = positions
    def total_value(self) -> float:
        return sum(p.market_value() for p in self.positions)
    # Add a helper method that aggregates market value for a single symbol.
    def value_by_symbol(self, symbol: str) -> float:
        return sum(
            p.market_value() for p in self.positions if p.symbol == symbol
        )

In [ ]:
portfolio = Portfolio(
    [Position("AAPL", 10, 180.0), Position("AAPL", 5, 182.0)]
)

In [ ]:
# Call the method to obtain the current value of all AAPL positions in the
# portfolio.
portfolio.value_by_symbol("AAPL")

# Designing for Responsibilities, Not Hierarchies


Prefer small components with clear responsibilities over deep class trees.


### Example: A Price Source Interface


Abstract interfaces let you swap in different price sources without changing
the caller.


In [ ]:
from abc import ABC, abstractmethod

In [ ]:
# Define an abstract base class with a single required method.
class PriceSource(ABC):
    @abstractmethod
    def get_price(self, symbol: str) -> float:
        # Document the contract for `.get_price()` in a docstring.
        """Return the latest price for the given symbol."""

In [ ]:
# Implement a concrete subclass that satisfies the interface.
class DictPriceSource(PriceSource):
    def __init__(self, prices: dict[str, float]) -> None:
        self.prices = prices
    # Store prices in a dictionary and implement `.get_price()` accordingly.
    def get_price(self, symbol: str) -> float:
        return self.prices[symbol]

In [ ]:
src = DictPriceSource({"AAPL": 180.0, "MSFT": 350.0})

In [ ]:
# Call the method through the concrete class; calling code can later depend on
# the `PriceSource` interface instead of a specific implementation.
src.get_price("MSFT")

### Encapsulation with Properties


Properties let you protect invariants while keeping an attribute-style
interface.


In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class SafePosition:
    symbol: str
    _qty: float
    price: float
    @property
    # Expose `qty` as a read-only-style property for callers; they
    # access it like
    # an attribute, not like a method.
    def qty(self) -> float:
        return self._qty
    @qty.setter
    # Enforce a simple invariant in the setter and reject negative
    # quantities with
    # a clear error.
    def qty(self, value: float) -> None:
        if value < 0:
            raise ValueError("quantity cannot be negative")
        self._qty = value
    # Implement `.market_value()` in terms of the encapsulated `_qty`
    # and `price`.
    def market_value(self) -> float:
        return self._qty * self.price

In [ ]:
pos = SafePosition("AAPL", 10, 180.0)

In [ ]:
# From the outside, the object behaves like a regular position with a `.qty`
# attribute and a `.market_value()` method.
pos.qty, pos.market_value()

In [ ]:
# Attempting to violate the invariant raises an exception instead of silently
# storing a bad value.
try:
    pos.qty = -5
except ValueError as exc:
    print(f"Caught expected error: {exc}")

### Inheritance and Polymorphism in Portfolios


Subclasses can reuse base behavior and only override what changes.


In [ ]:
from dataclasses import dataclass

In [ ]:
@dataclass
class Position:
    symbol: str
    qty: float
    price: float  # price in the asset's local currency
    def market_value(self) -> float:
        return self.qty * self.price

In [ ]:
@dataclass
# Define `FXPosition` as a subclass of `Position` that records both the
# reporting currency and the FX rate from the local currency into that
# reporting currency.
class FXPosition(Position):
    reporting_ccy: str
    fx_to_reporting: float
    # Override `.market_value()` to convert the local-currency value returned by
    # the base class into the reporting currency while still reusing the base-
    # class implementation via `super()`.
    def market_value(self) -> float:
        local_value = super().market_value()
        return local_value * self.fx_to_reporting

In [ ]:
fx_pos = FXPosition(
    "BOND_EUR",
    100_000,
    1.10,
    reporting_ccy="USD",
    fx_to_reporting=1.08,
)


In [ ]:
# The subclass can be used wherever a `Position` is expected, but it
# contributes a different notion of value: here, a EUR‑denominated bond
# expressed in USD.
fx_pos.market_value()

In [ ]:
class Portfolio:
    def __init__(self, positions: list[Position]) -> None:
        self.positions = positions
    def total_value(self) -> float:
        return sum(p.market_value() for p in self.positions)

In [ ]:
portfolio = Portfolio(
    [
        Position("AAPL", 10, 180.0),
        FXPosition(
            "BOND_EUR",
            100_000,
            1.10,
            reporting_ccy="USD",
            fx_to_reporting=1.08,
        ),
    ]
)


In [ ]:
# The `Portfolio` treats both `Position` and `FXPosition` uniformly by calling
# `.market_value()` on each; polymorphism takes care of the details.
portfolio.total_value()

### Combining Orthogonal Classes


Composition keeps the valuation engine focused on one responsibility.


In [ ]:
# The `ValuationEngine` class depends only on the `PriceSource` interface, not
# on a specific implementation.
class ValuationEngine:
    def __init__(self, price_source: PriceSource) -> None:
        self.price_source = price_source
    # The `.price_position()` method uses the price source to construct a
    # `Position` with a current price.
    def price_position(self, symbol: str, qty: float) -> Position:
        price = self.price_source.get_price(symbol)
        return Position(symbol, qty, price)

In [ ]:
engine = ValuationEngine(src)

In [ ]:
# Calling code can plug in different `PriceSource` subclasses (for example,
# live market data or cached files) without changing the valuation logic.
engine.price_position("AAPL", 10).market_value()

# Where We Are Heading Next


The object-oriented patterns in this chapter make it easier to
build the larger systems that appear later in the book.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
